# Earnings Dates
Past earnings dates with AM/PM (yfinance) and the next upcoming date (Finnhub) → `Reports/earnings_date.csv`
(`Symbol, Earnings Date, Time`). Upcoming AM/PM = Finnhub's announced time; when not announced yet, predicted as the stock's usual
time over its past reports.

- The file is **merged**, not replaced: past dates already on file are kept (the backtests use them), newly fetched past dates
  win on conflicts, and stale "upcoming" dates that are now in the past but not confirmed by yfinance are dropped.
- **Modes** (env `PIPELINE_EARNINGS_MODE`, default `online` when run by hand): `online` (yfinance for all + ≈97 Finnhub calls),
  `offline` (no network: just normalizes/deduplicates the existing file), `sample` (1–2 symbols, writes to `Reports/cache/`).

### 1. Setup
Paths, symbols and the mode (`PIPELINE_EARNINGS_MODE`: online / offline / sample; offline makes no network calls).

In [1]:
import json
import os
import time
import warnings
from datetime import timedelta

import pandas as pd
from dotenv import load_dotenv

import sector_mapping
from sector_mapping import REPORTS_DIR, stock_symbols

load_dotenv(os.path.join(sector_mapping.PROJECT_ROOT, ".env"))
warnings.filterwarnings("ignore", module="yfinance")
MODE = os.getenv("PIPELINE_EARNINGS_MODE", "online").lower()
assert MODE in {"online", "offline", "sample"}, MODE
TODAY = pd.Timestamp.now(tz="America/New_York").date()
EARNINGS_TIME = {"bmo": "AM", "amc": "PM", "dmh": "Midday"}
OUT_CSV = os.path.join(REPORTS_DIR, "earnings_date.csv")
SYMBOLS = [s.strip().upper() for s in os.getenv("PIPELINE_SAMPLE_SYMBOLS", "NVDA").split(",")][:2] if MODE == "sample" else stock_symbols
COLUMNS = ["Symbol", "Earnings Date", "Time"]
CALLS = {"finnhub": 0}   # request attempts, for the run summary
print(f"Mode: {MODE} | symbols: {len(SYMBOLS)} | today (ET) {TODAY}")

Mode: online | symbols: 97 | today (ET) 2026-09-25


### 2. Past earnings dates (yfinance, with AM/PM)

Two fetchers, two sources: `fetch_past_earnings` uses yfinance (free, no key) for the last 2 years of actual report dates;
`fetch_next_earnings` uses Finnhub's earnings calendar (API key) for the next upcoming date per symbol.

- **AM/PM inference (yfinance):** the timestamp's hour in ET decides it — before noon → AM (before-market report),
  noon or later → PM (after-market report). This matters because the strategy blocks buys/sells around earnings.
- **Finnhub retries:** 3 attempts per symbol with linear backoff (5s, 10s); the symbol lands in `failed_next` only if all 3 fail.
- Both return `(df, failed_list)` so the caller can log per-symbol failures without aborting the whole run.

In [2]:
def _is_auth_error(e):
    """True for HTTP 401/403 auth failures (finnhub raises these with a status attr); retrying is pointless."""
    for attr in ("status", "status_code", "code"):
        if getattr(e, attr, None) in (401, 403, "401", "403"):
            return True
    msg = str(e).lower()
    return "401" in msg or "403" in msg or "unauthorized" in msg or "forbidden" in msg


def fetch_past_earnings(symbols, years_back=2):
    import yfinance as yf
    cutoff = TODAY - timedelta(days=years_back * 365)
    rows, failed = [], []
    for i, symbol in enumerate(symbols):
        dates = None
        for attempt in range(3):                       # retry transient yfinance errors
            try:
                dates = yf.Ticker(symbol).get_earnings_dates(limit=12)
                break
            except Exception:
                if attempt < 2:
                    time.sleep(2 ** attempt)            # 1s, 2s backoff
        if i < len(symbols) - 1:
            time.sleep(0.3)                            # be gentle on the free API
        if dates is None or dates.empty:
            failed.append(symbol)
            continue
        idx = dates.index if dates.index.tz is not None else dates.index.tz_localize("America/New_York")
        rows += [{"Symbol": symbol, "Earnings Date": d.date(), "Time": "AM" if d.hour < 12 else "PM"}
                 for d in idx.tz_convert("America/New_York") if cutoff <= d.date() < TODAY]
    return pd.DataFrame(rows, columns=COLUMNS), failed


def fetch_next_earnings(symbols, days_ahead=100):
    import finnhub
    client = finnhub.Client(api_key=os.environ["FINNHUB_API_KEY"])
    end = (TODAY + timedelta(days=days_ahead)).isoformat()
    rows, failed = [], []
    for symbol in symbols:
        for attempt in range(3):
            try:
                CALLS["finnhub"] += 1
                calendar = client.earnings_calendar(symbol=symbol, _from=TODAY.isoformat(), to=end).get("earningsCalendar", [])
                break
            except Exception as e:
                if _is_auth_error(e):
                    raise                              # bad key: retrying all symbols is pointless
                if attempt == 2:
                    calendar = None
                    failed.append(symbol)
                else:
                    time.sleep(5 * (attempt + 1))
        time.sleep(1.1)  # free tier: 60 calls/min
        events = [(e["date"], e.get("hour") or "") for e in (calendar or []) if e.get("date", "") >= TODAY.isoformat()]
        upcoming = sorted(events, key=lambda e: (e[0], e[1]))  # None hours -> "" so mixed types never TypeError
        if upcoming:
            rows.append({"Symbol": symbol, "Earnings Date": upcoming[0][0], "Time": EARNINGS_TIME.get(upcoming[0][1])})
    return pd.DataFrame(rows, columns=COLUMNS), failed

### 3. Next earnings date (Finnhub), merge with the saved calendar and save `Reports/earnings_date.csv`

The saved file is **merged, not replaced** — the backtests depend on its history, so past dates are never thrown away:

- **Existing past rows are kept**; freshly fetched yfinance rows win when both have the same symbol/date (dedup keeps first).
- **Stale upcoming dates are dropped:** if a saved "upcoming" date is now in the past but yfinance doesn't list it,
  the date was moved or wrong — keeping it would block trading forever, so it's removed.
- **Missing Finnhub AM/PM is backfilled** from the symbol's usual report time (the mode of its past AM/PM values),
  because Finnhub often announces the date before the time.
- Safety: in `online` mode, if *both* fetches come back empty the run raises instead of overwriting the file with nothing.

In [3]:
existing = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame(columns=COLUMNS)
existing["Earnings Date"] = pd.to_datetime(existing["Earnings Date"], errors="coerce")

if MODE == "offline":
    past_df, next_df = pd.DataFrame(columns=COLUMNS), pd.DataFrame(columns=COLUMNS)
    failed_past, failed_next = [], []
else:
    past_df, failed_past = fetch_past_earnings(SYMBOLS)
    next_df, failed_next = fetch_next_earnings(SYMBOLS)
    print(f"yfinance past rows: {len(past_df)} (failed: {failed_past}); Finnhub upcoming: {len(next_df)} (failed: {failed_next})")
    print(f"API calls: finnhub={CALLS['finnhub']}")
    if os.getenv("PIPELINE_CALLS_FILE"):                   # run_all.py adds these up for its summary
        with open(os.environ["PIPELINE_CALLS_FILE"], "w") as f:
            f.write(json.dumps(CALLS) + "\n")
    if MODE == "online" and past_df.empty and next_df.empty:
        raise RuntimeError("No earnings data fetched - keeping the previous file")

for d in (past_df, next_df):
    d["Earnings Date"] = pd.to_datetime(d["Earnings Date"], errors="coerce")

# usual AM/PM per symbol from all known past reports (new + existing)
known_past = pd.concat([past_df, existing[existing["Earnings Date"] < pd.Timestamp(TODAY)]])
usual_time = known_past.dropna(subset=["Time"]).groupby("Symbol")["Time"].agg(lambda t: t.mode().iloc[0])
next_df["Time"] = next_df["Time"].fillna(next_df["Symbol"].map(usual_time))

# merge: keep existing past rows (fresh yfinance rows win); drop existing future rows ONLY for symbols
# that actually answered - a failed fetch must not wipe known future dates (earnings block)
refreshed = set(past_df["Symbol"]) | set(next_df["Symbol"])
failed = set(failed_past) | set(failed_next)
answered = refreshed - failed
old_keep = existing[(existing["Earnings Date"] < pd.Timestamp(TODAY)) | ~existing["Symbol"].isin(answered)]
if len(past_df):
    # An old "upcoming" date that is now in the past but yfinance does not list was moved or wrong -> drop it.
    # (Keeping it would block trading on that stock forever.)
    confirmed = set(zip(past_df["Symbol"], past_df["Earnings Date"]))
    window_start = pd.Timestamp(TODAY - timedelta(days=90))
    old_keys = pd.Series(list(zip(old_keep["Symbol"], old_keep["Earnings Date"])), index=old_keep.index)
    from_refreshed = old_keep["Symbol"].isin(set(past_df["Symbol"]))
    in_window = old_keep["Earnings Date"].between(window_start, pd.Timestamp(TODAY))
    old_keep = old_keep[~(from_refreshed & in_window & ~old_keys.isin(confirmed))]
earnings_df = pd.concat([past_df, next_df, old_keep], ignore_index=True)
earnings_df["Symbol"] = earnings_df["Symbol"].astype(str).str.strip().str.upper()
earnings_df = (earnings_df.dropna(subset=["Earnings Date"])
               .drop_duplicates(["Symbol", "Earnings Date"], keep="first")
               .sort_values(["Symbol", "Earnings Date"], ascending=[True, False]).reset_index(drop=True))[COLUMNS]

if MODE == "sample":
    path = os.path.join(REPORTS_DIR, "cache", "sample_earnings.csv")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print(f"Sample mode: wrote {path} only")
else:
    earnings_df.to_csv(OUT_CSV, index=False)
    print(f"✅ earnings_date.csv: {len(earnings_df)} rows, {earnings_df['Symbol'].nunique()} symbols")
earnings_df[earnings_df["Earnings Date"] >= pd.Timestamp(TODAY)].sort_values("Earnings Date").head(10)

QQQ: No earnings dates found, symbol may be delisted


yfinance past rows: 758 (failed: ['QQQ']); Finnhub upcoming: 96 (failed: [])
API calls: finnhub=97
✅ earnings_date.csv: 853 rows, 96 symbols


Symbol Earnings Date Time
523     MU    2026-09-30   PM
799    UNH    2026-10-13   AM
171      C    2026-10-13   AM
81    APLD    2026-10-14   PM
310     FCX    2026-10-15   AM
675     SLB    2026-10-16   AM
424     LMT    2026-10-19   AM
835     VRT    2026-10-20   AM
817     URI    2026-10-20   PM
352     HAL    2026-10-20   AM